# Project 9 — Traffic Forecasting (Report Template)

Concise reproducible notebook: data load, EDA, feature engineering, model training, predictions, and report artifacts.

In [ ]:
# Setup
import sys, os
from pathlib import Path
repo_root = Path('..').resolve()
sys.path.insert(0, str(repo_root))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
%matplotlib inline
try:
    plt.style.use('seaborn')
except Exception:
    pass
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
# ensure reports folder exists
(repo_root / 'reports').mkdir(exist_ok=True)

In [ ]:
# Data loading
train = pd.read_csv(repo_root / 'data' / 'train.csv', parse_dates=['DateTime'])
test = pd.read_csv(repo_root / 'data' / 'test.csv', parse_dates=['DateTime'])
print('train shape:', train.shape)
print('test shape :', test.shape)
train.head()

In [ ]:
# Missing values and duplicates
print(train.isna().sum())
print('duplicates in train:', train.duplicated().sum())

In [ ]:
# Basic EDA: distribution and boxplot
fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.histplot(train['Vehicles'], bins=60, kde=True, ax=ax[0])
ax[0].set_title('Distribution of Vehicles')
sns.boxplot(x='Junction', y='Vehicles', data=train, ax=ax[1])
ax[1].set_title('Traffic by Junction')
plt.tight_layout()
plt.savefig(repo_root / 'reports' / 'eda_distribution_box.png')
plt.show()

In [ ]:
# Time-based plots: hourly and weekday averages
train['Hour'] = train['DateTime'].dt.hour
train['DayOfWeek'] = train['DateTime'].dt.day_name()
hourly = train.groupby('Hour')['Vehicles'].mean()
weekday = train.groupby('DayOfWeek')['Vehicles'].mean().reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
fig, ax = plt.subplots(1,2, figsize=(12,4))
hourly.plot(ax=ax[0], marker='o')
ax[0].set_title('Average Traffic by Hour')
weekday.plot(kind='bar', ax=ax[1], color='C2')
ax[1].set_title('Average Traffic by Day of Week')
plt.tight_layout()
plt.savefig(repo_root / 'reports' / 'eda_hour_weekday.png')
plt.show()

In [ ]:
# Correlation heatmap on selected features
feat = train.copy()
feat['Year'] = feat['DateTime'].dt.year
feat['Month'] = feat['DateTime'].dt.month
feat['Day'] = feat['DateTime'].dt.day
feat['Hour'] = feat['DateTime'].dt.hour
corr_cols = ['Vehicles','Year','Month','Day','Hour']
corr = feat[corr_cols].corr()
plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True, cmap='RdBu_r')
plt.title('Correlation Heatmap')
plt.savefig(repo_root / 'reports' / 'eda_correlation.png')
plt.show()

In [ ]:
# Feature engineering using functions from train_models.py
import importlib.util
spec = importlib.util.spec_from_file_location('train_models', str(repo_root / 'train_models.py'))
tm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tm)
train_feat = tm.add_time_features(train)
train_feat = tm.add_lag_features(train_feat)
train_feat = train_feat.dropna().reset_index(drop=True)
train_feat.columns.tolist()[:30]

In [ ]:
# Train/validation split and model training
tr, val = tm.time_based_split(train_feat)
FEATURE_COLS = tm.FEATURE_COLS
TARGET = tm.TARGET_COL
X_train, y_train = tr[FEATURE_COLS], tr[TARGET]
X_val, y_val = val[FEATURE_COLS], val[TARGET]
models = {}
results = []
# Linear Regression
lr = LinearRegression(); lr.fit(X_train, y_train); models['Linear Regression']=lr;
pred = lr.predict(X_val); results.append({'model':'Linear Regression','MAE':mean_absolute_error(y_val,pred),'RMSE':np.sqrt(mean_squared_error(y_val,pred)),'R2':r2_score(y_val,pred)})
# Decision Tree
dt = DecisionTreeRegressor(max_depth=12, random_state=42); dt.fit(X_train,y_train); models['Decision Tree']=dt;
pred = dt.predict(X_val); results.append({'model':'Decision Tree','MAE':mean_absolute_error(y_val,pred),'RMSE':np.sqrt(mean_squared_error(y_val,pred)),'R2':r2_score(y_val,pred)})
# Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=3, n_jobs=1, random_state=42); rf.fit(X_train,y_train); models['Random Forest']=rf;
pred = rf.predict(X_val); results.append({'model':'Random Forest','MAE':mean_absolute_error(y_val,pred),'RMSE':np.sqrt(mean_squared_error(y_val,pred)),'R2':r2_score(y_val,pred)})
results_df = pd.DataFrame(results).sort_values('RMSE')
results_df

In [ ]:
# Refit best model on full train and save
best_name = results_df.iloc[0]['model']
builders = {
    'Linear Regression': lambda: LinearRegression(),
    'Decision Tree': lambda: DecisionTreeRegressor(max_depth=12, random_state=42),
    'Random Forest': lambda: RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=3, n_jobs=1, random_state=42),
}
final_model = builders[best_name]()
X_full, y_full = train_feat[FEATURE_COLS], train_feat[TARGET]
final_model.fit(X_full, y_full)
joblib.dump(final_model, repo_root / 'models' / 'traffic_prediction_model_from_notebook.pkl')
# feature importances plot if available
if hasattr(final_model, 'feature_importances_'):
    imp = pd.Series(final_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    imp.plot(kind='barh', figsize=(6,4))
    plt.title('Feature Importances')
    plt.tight_layout()
    plt.savefig(repo_root / 'reports' / 'feature_importances_notebook.png')
    plt.show()
best_name

In [ ]:
# Predict on test set (autoregressive) and save submission
from collections import deque
test_feat = tm.add_time_features(test.sort_values(['Junction','DateTime']).reset_index(drop=True))
all_preds = []
for j, test_group in test_feat.groupby('Junction'):
    train_tail = train[train['Junction']==j].sort_values('DateTime')['Vehicles'].tail(24).tolist()
    history = deque(train_tail, maxlen=24)
    for _, row in test_group.sort_values('DateTime').iterrows():
        lag1 = history[-1]
        lag24 = history[0] if len(history)==24 else history[-1]
        rolling_mean = float(np.mean(history))
        feat_row = pd.DataFrame([{
            'Junction': j, 'Year': row['Year'], 'Month': row['Month'], 'Day': row['Day'], 'Hour': row['Hour'],
            'DayOfWeek': row['DayOfWeek'], 'Weekend': row['Weekend'], 'WeekNumber': row['WeekNumber'], 'Quarter': row['Quarter'],
            'Season': row['Season'], 'MonthStart': row['MonthStart'], 'MonthEnd': row['MonthEnd'], 'QuarterStart': row['QuarterStart'],
            'QuarterEnd': row['QuarterEnd'], 'IsHoliday': row['IsHoliday'], 'Lag1': lag1, 'Lag24': lag24, 'RollingMean': rolling_mean
        }])
        pred = max(round(final_model.predict(feat_row[FEATURE_COLS])[0]), 1)
        history.append(pred)
        all_preds.append({'ID': row['ID'], 'Vehicles': int(pred)})
submission = pd.DataFrame(all_preds).sort_values('ID').reset_index(drop=True)
submission.to_csv(repo_root / 'data' / 'submission.csv', index=False)
submission.head()

## Summary
- Best model selected and retrained on full data.
- Submission written to `data/submission.csv`.
- Artifacts saved under `reports/` and `models/`.

Next steps: tune model hyperparameters, add calendar events, and evaluate per-junction performance.